In [1]:
from spending_tracker.database import supabase_client
from spending_tracker.database import transactions
from spending_tracker.data_ingestion import csv_parsing
from pathlib import Path

In [2]:
db_client = supabase_client.load_supabase_client()

In [3]:
processed_transactions_data_home = Path().resolve().parent / "data" / "transactions" / "processed"
transaction_metadata_home = Path().resolve().parent / "data" / "transactions_metadata"
print(processed_transactions_data_home)

/Users/taj/Documents/Spending-Tracker-Agent/data/transactions/processed


In [4]:
transactions_df = csv_parsing.load_csv_in_batch(processed_transactions_data_home,
                                                account_id=str)

transactions_metadata_df = csv_parsing.load_csv_in_batch(transaction_metadata_home)

In [5]:
transactions_df.head(10)

,event_id,date,month,account_id,user,bank,merchant,other_details,type,amount
0,4dfe72a057bb9e273208e3470ea91f0a64779283185e80...,2026-07-07,July-26,03076275,Joint,Nationwide,TAJ PATEL,NaN,NaN,500.00
1,897b964f7d4d2e001d6149f152bec20e3413e690eae85c...,2026-07-08,July-26,03076275,Joint,Nationwide,TESCO STORES 5620 LONDON GB,APPLEPAY 4914,NaN,-3.80
2,c70e40fe2579c0e27e76a1d16f8a7d91d3b5f4dd16b485...,2026-07-09,July-26,03076275,Joint,Nationwide,KAMLA PATEL,NaN,NaN,100.00
3,2f052381c3482ca02d6bd29a6ab5e1c764f2cae4414962...,2026-07-09,July-26,03076275,Joint,Nationwide,PRIYA CHAUHAN,NaN,NaN,190.00
4,f6a9bb8f21bc3aac5928d8395d55a38ec44b4ae25aa3d2...,2026-07-09,July-26,03076275,Joint,Nationwide,071490 77608825 Withdrawal 09 July 2026,NaN,NaN,-290.00
5,959b86f5d0ae066ddbefe59c6ba07d42e84f5bf2e7f678...,2026-07-10,July-26,03076275,Joint,Nationwide,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,APPLEPAY 4914,NaN,-9.36
6,2c25293fb239d900915db5245fc12bca371c76276e286d...,2026-07-11,July-26,03076275,Joint,Nationwide,ASDA SUPERSTORE COLINDALE GB,APPLEPAY 9160,NaN,-5.00
7,eae2b36adcf5123c180c574db84cd29b56170cbf07958c...,2026-07-11,July-26,03076275,Joint,Nationwide,FUNKY PIGEON BRISTOL GB,APPLEPAY 9160,NaN,-7.29
8,13d5d5ad99a408268ed354a262e3799425b332d0b54be2...,2026-07-11,July-26,03076275,Joint,Nationwide,MARKS&SPENCER PLC LONDON GB,APPLEPAY 9160,NaN,-7.43
9,c10b670b09fc44c817cbbe8359d89a61e87142dcf02045...,2026-07-11,July-26,03076275,Joint,Nationwide,M&S SALFORD INTERNET GB,APPLEPAY 5176,NaN,-25.00


In [6]:
transactions_metadata_df.head()

,event_id,llm_merchant,llm_category,manual_merchant,manual_category,manual_amount,hidden
0,4dfe72a057bb9e273208e3470ea91f0a64779283185e80...,TAJ PATEL,NaN,NaN,NaN,NaN,False
1,897b964f7d4d2e001d6149f152bec20e3413e690eae85c...,TESCO STORES 5620 LONDON GB,NaN,NaN,NaN,NaN,False
2,c70e40fe2579c0e27e76a1d16f8a7d91d3b5f4dd16b485...,KAMLA PATEL,NaN,NaN,NaN,NaN,False
3,2f052381c3482ca02d6bd29a6ab5e1c764f2cae4414962...,PRIYA CHAUHAN,NaN,NaN,NaN,NaN,False
4,f6a9bb8f21bc3aac5928d8395d55a38ec44b4ae25aa3d2...,071490 77608825 Withdrawal 09 July 2026,NaN,NaN,NaN,NaN,False


In [7]:
transactions_df.shape

(181, 10)

In [8]:
transactions_metadata_df.shape

(181, 7)

In [9]:
transactions.upload_transactions_to_db(transactions_df, 
                                       db_client, 
                                       table_name="transactions")

APIResponse(data=[{'event_id': '4dfe72a057bb9e273208e3470ea91f0a64779283185e80a84672c3607b9a5895', 'date': '2026-07-07', 'month': 'July-26', 'account_id': '03076275', 'user': 'Joint', 'bank': 'Nationwide', 'merchant': 'TAJ PATEL', 'other_details': None, 'type': None, 'amount': 500, 'updated_at': '2026-08-26T15:26:50.48158+00:00'}, {'event_id': '897b964f7d4d2e001d6149f152bec20e3413e690eae85c64cf552a453c7a11bc', 'date': '2026-07-08', 'month': 'July-26', 'account_id': '03076275', 'user': 'Joint', 'bank': 'Nationwide', 'merchant': 'TESCO STORES 5620 LONDON GB', 'other_details': 'APPLEPAY 4914', 'type': None, 'amount': -3.8, 'updated_at': '2026-08-26T15:26:50.48158+00:00'}, {'event_id': 'c70e40fe2579c0e27e76a1d16f8a7d91d3b5f4dd16b485cfd79d7614aa624131', 'date': '2026-07-09', 'month': 'July-26', 'account_id': '03076275', 'user': 'Joint', 'bank': 'Nationwide', 'merchant': 'KAMLA PATEL', 'other_details': None, 'type': None, 'amount': 100, 'updated_at': '2026-08-26T15:26:50.48158+00:00'}, {'eve

In [11]:
transactions.upload_transactions_to_db(transactions_metadata_df, 
                                       db_client, 
                                       table_name="transactions_metadata", 
                                       exclude_columns=["manual_merchant", "manual_category", "manual_amount", "hidden"])

APIResponse(data=[{'event_id': '4dfe72a057bb9e273208e3470ea91f0a64779283185e80a84672c3607b9a5895', 'llm_merchant': 'TAJ PATEL', 'llm_category': None, 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': None, 'updated_at': '2026-08-26T15:26:59.667221+00:00'}, {'event_id': '897b964f7d4d2e001d6149f152bec20e3413e690eae85c64cf552a453c7a11bc', 'llm_merchant': 'TESCO STORES 5620 LONDON GB', 'llm_category': None, 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': None, 'updated_at': '2026-08-26T15:26:59.667221+00:00'}, {'event_id': 'c70e40fe2579c0e27e76a1d16f8a7d91d3b5f4dd16b485cfd79d7614aa624131', 'llm_merchant': 'KAMLA PATEL', 'llm_category': None, 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': None, 'updated_at': '2026-08-26T15:26:59.667221+00:00'}, {'event_id': '2f052381c3482ca02d6bd29a6ab5e1c764f2cae441496204d9092b0a0a440843', 'llm_merchant': 'PRIYA CHAUHAN', 'llm_category': None, 'manual_me